# Project Additional Materials — Spatial Dataset Preparation

- Student ID: 10841269  
- Course Code: DATA70132  
- Academic Year: 2024–25  

**Environment:** See `README` and `ERP_Environment_2025.yaml`.  
**Reproduction:** Run this notebook top to bottom in the same folder as `All_UKonly_daily_cleaned_ymd.nc`.  

This notebook prepares the spatial dataset for modelling.


## Step 0 — Import required libraries

In [ ]:
# Import libraries
import xarray as xr
import numpy as np
import pandas as pd
from flaml import AutoML
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import random
import os
import warnings, os, json
warnings.filterwarnings("ignore")

## Step 1 — Define functions

In [ ]:
def clone_models(rename_map):
    for src, dst in rename_map.items():
        if not os.path.exists(src):
            print(f"[skip] {src} not found")
            continue
        with open(src, "rb") as f:
            model = pickle.load(f)
        with open(dst, "wb") as f:
            pickle.dump(model, f, pickle.HIGHEST_PROTOCOL)
        print(f"[ok]  {src} -> {dst}")

def day_spatial_q95(arr_day):  # arr_day: (sim, P_day)
    with np.errstate(invalid="ignore"):
        return np.nanquantile(arr_day, 0.95, axis=1).astype(np.float32)
    
def build_xy_memmap(points_idx, prefix, batch_points=20000, dtype=np.float32):
    n_rows = points_idx.size * n_sim
    X_mm_path = f"{"splits_mm"}/{prefix}_X.mm"
    y_mm_path = f"{"splits_mm"}/{prefix}_y.mm"
    X_mm = np.memmap(X_mm_path, mode="w+", dtype=dtype, shape=(n_rows, N_FEAT))
    y_mm = np.memmap(y_mm_path, mode="w+", dtype=dtype, shape=(n_rows,))

    targ_view = ds["TREFMXAV_U"].transpose("point","sim")
    feat_view = {v: ds[v].transpose("point","sim") for v in VARS_X}

    write = 0; B = int(batch_points)
    for start in range(0, points_idx.size, B):
        idx = points_idx[start:start+B]
        y_block = targ_view.isel(point=idx).values.reshape(-1)

        cols = [feat_view[v].isel(point=idx).values.reshape(-1) for v in VARS_X]
        if True:
            cols += [np.repeat(lat[idx], n_sim), np.repeat(lon[idx], n_sim)]
        X_block = np.column_stack(cols)

        mask = np.isfinite(y_block)
        for j in range(X_block.shape[1]): mask &= np.isfinite(X_block[:, j])
        if not mask.all():
            y_block = y_block[mask]; X_block = X_block[mask]

        nb = y_block.shape[0]
        X_mm[write:write+nb, :] = X_block.astype(dtype, copy=False)
        y_mm[write:write+nb]    = y_block.astype(dtype, copy=False)
        write += nb

        del y_block, X_block, cols
        if ((start // B) + 1) % 50 == 0:
            print(f"[{prefix}] rows: {write}/{n_rows}")

    X_mm.flush(); y_mm.flush()
    del X_mm; del y_mm

    meta = {
        "features": FEATURES, "target": "TREFMXAV_U",
        "rows": int(write), "cols": int(N_FEAT),
        "dtype": np.dtype(dtype).name, "sims": int(n_sim),
        "memmap_X": X_mm_path, "memmap_y": y_mm_path,
        "coords_in_X": bool(True), "split_name": prefix
    }
    with open(f"{"splits_mm"}/{prefix}_meta.json", "w") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f"[{prefix}] saved →", X_mm_path, y_mm_path, "rows:", write)

## Step 2 — Load dataset and separate sets
Separate the dataset into top 30%, middle 40%, and bottom 30% groups for the models.  

In [ ]:
# Config
VARS_X  = ["FLNS","FSNS","PRECT","PRSN","QBOT","TREFHT","UBOT","VBOT"]
TARGET  = "TREFMXAV_U"
P_TOP, P_MID, P_BOTTOM = 0.30, 0.40, 0.30

# Load
ds = xr.open_dataset("All_UKonly_daily_cleaned_ymd.nc", chunks={"sim": 1, "point": 20000})
ds = ds.astype({v: "float32" for v in VARS_X + ["TREFMXAV_U"]})
da = ds["TREFMXAV_U"]  # (sim, point); 'time' is a coord on point
n_point, n_sim = ds.sizes["point"], ds.sizes["sim"]
lat = ds["lat"].values.astype("float64")
lon = ds["lon"].values.astype("float64")
time_coord = ds["time"].values

# Daily q95 voting
order = np.argsort(time_coord)
sorted_time = time_coord[order]
is_new = np.empty(sorted_time.size, bool); is_new[0] = True
is_new[1:] = sorted_time[1:] != sorted_time[:-1]
starts = np.flatnonzero(is_new)
ends = np.r_[starts[1:], sorted_time.size]
unique_days = sorted_time[starts]

votes_pt  = np.zeros(n_point, np.int32)
ratio_sum = np.zeros(n_point, np.float32)
hit_sum   = np.zeros(n_point, np.int32)

targ_view = da.transpose("sim", "point")
for di in range(len(unique_days)):
    idx = order[starts[di]:ends[di]]
    day_vals = targ_view.isel(point=idx).values
    thr  = day_spatial_q95(day_vals)
    mask = np.greater_equal(day_vals, thr[:, None], where=np.isfinite(day_vals))
    hits = mask.sum(axis=0).astype(np.int32)

    votes_pt[idx] += hits
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(mask, day_vals / thr[:, None], 0.0).astype(np.float32)
    ratio_sum[idx] += ratio.sum(axis=0).astype(np.float32)
    hit_sum[idx]   += hits

    if (di + 1) % 500 == 0:
        print(f"Processed days: {di+1}/{len(unique_days)}")

rk_pt = np.zeros(n_point, np.float32)
nz = hit_sum > 0
rk_pt[nz] = (ratio_sum[nz] / hit_sum[nz]).astype(np.float32)

# Aggregate to (lat, lon)
lat_r = np.round(lat, 5)
lon_r = np.round(lon, 5)
keys = np.core.records.fromarrays([lat_r, lon_r], names="lat,lon")
uniq_locs, loc_idx = np.unique(keys, return_inverse=True)
N_loc = uniq_locs.size

cnt_loc = np.bincount(loc_idx, weights=votes_pt.astype(np.float64), minlength=N_loc).astype(np.int64)
rk_sum  = np.bincount(loc_idx, weights=rk_pt.astype(np.float64),    minlength=N_loc).astype(np.float64)
cnt_num = np.bincount(loc_idx, minlength=N_loc).astype(np.int64)

rk_loc = np.zeros(N_loc, np.float64)
mask_num = cnt_num > 0
rk_loc[mask_num] = rk_sum[mask_num] / cnt_num[mask_num]

df_split_loc = (
    pd.DataFrame({
        "lat": uniq_locs["lat"].astype(np.float64),
        "lon": uniq_locs["lon"].astype(np.float64),
        "cnt": cnt_loc,
        "rk":  rk_loc
    })
    .sort_values(["cnt","rk"], ascending=[True, True])
    .reset_index(drop=True)
)

# Split to bottom30 / mid40 / top30
n_all = len(df_split_loc)
n_bottom = int(round(P_BOTTOM * n_all))
n_top    = int(round(P_TOP    * n_all))
idx_bottom = df_split_loc.index[:n_bottom]
idx_top    = df_split_loc.index[-n_top:]
idx_mid    = df_split_loc.index[n_bottom: n_all - n_top]

split = pd.Series(index=df_split_loc.index, dtype="object")
split.loc[idx_top]    = "top30"
split.loc[idx_mid]    = "mid40"
split.loc[idx_bottom] = "bottom30"
df_split_loc["split"] = split.values

print("Location-level split:",
      "top30",    (df_split_loc["split"]=="top30").sum(),
      "mid40",    (df_split_loc["split"]=="mid40").sum(),
      "bottom30", (df_split_loc["split"]=="bottom30").sum())

# Map back to point indices
loc2split = dict(zip(zip(df_split_loc["lat"].values, df_split_loc["lon"].values),
                     df_split_loc["split"].values))
pt_split = np.empty(n_point, dtype=object)
for i in range(n_point):
    pt_split[i] = loc2split.get((lat_r[i], lon_r[i]), None)

top_points    = np.flatnonzero(pt_split == "top30").astype(np.int64)
mid_points    = np.flatnonzero(pt_split == "mid40").astype(np.int64)
bottom_points = np.flatnonzero(pt_split == "bottom30").astype(np.int64)

print("Point counts:", "top30", top_points.size, "mid40", mid_points.size, "bottom30", bottom_points.size)

# Write X/y (memmap, batched)
os.makedirs(splits_mm, exist_ok=True)
FEATURES = VARS_X + (["lat","lon"] if True else [])
N_FEAT   = len(FEATURES)

build_xy_memmap(top_points,    "top30",    20000, np.float32)
build_xy_memmap(mid_points,    "mid40",    20000, np.float32)
build_xy_memmap(bottom_points, "bottom30", 20000, np.float32)

print("Done.")

Processed days: 500/27374
Processed days: 1000/27374
Processed days: 1500/27374
Processed days: 2000/27374
Processed days: 2500/27374
Processed days: 3000/27374
Processed days: 3500/27374
Processed days: 4000/27374
Processed days: 4500/27374
Processed days: 5000/27374
Processed days: 5500/27374
Processed days: 6000/27374
Processed days: 6500/27374
Processed days: 7000/27374
Processed days: 7500/27374
Processed days: 8000/27374
Processed days: 8500/27374
Processed days: 9000/27374
Processed days: 9500/27374
Processed days: 10000/27374
Processed days: 10500/27374
Processed days: 11000/27374
Processed days: 11500/27374
Processed days: 12000/27374
Processed days: 12500/27374
Processed days: 13000/27374
Processed days: 13500/27374
Processed days: 14000/27374
Processed days: 14500/27374
Processed days: 15000/27374
Processed days: 15500/27374
Processed days: 16000/27374
Processed days: 16500/27374
Processed days: 17000/27374
Processed days: 17500/27374
Processed days: 18000/27374
Processed da